<a href="https://colab.research.google.com/github/HariniAnandkumar/neuromorphic-ai-research/blob/main/03_hybrid_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4


In [ ]:
!pip install snntorch -q

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import snntorch as snn
import time
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
transform = transforms.ToTensor()

trainset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)

testset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

In [ ]:
num_steps = 10
beta = 0.95

class Hybrid(nn.Module):

    def __init__(self):
        super().__init__()

        # ANN feature extractor
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.relu1 = nn.ReLU()

        self.conv2 = nn.Conv2d(32, 64, 3)
        self.relu2 = nn.ReLU()

        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()

        # Spiking head
        self.fc1 = nn.Linear(9216, 128)
        self.lif1 = snn.Leaky(beta=beta)

        self.fc2 = nn.Linear(128, 10)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x):   # ← must be indented inside class

        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.pool(x)
        x = self.flatten(x)

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        spk1_rec = []
        spk2_rec = []

        for step in range(num_steps):
            cur1 = self.fc1(x)
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk1_rec.append(spk1)
            spk2_rec.append(spk2)

        return torch.stack(spk1_rec), torch.stack(spk2_rec)
model = Hybrid().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 10
start_time = time.time()

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        spk1, spk2 = model(images)
        output_sum = spk2.sum(dim=0)


        loss = criterion(output_sum, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(trainloader):.4f}")

train_time = time.time() - start_time
print("Training Time (sec):", train_time)

Epoch 1/10 | Loss: 0.1682
Epoch 2/10 | Loss: 0.0370
Epoch 3/10 | Loss: 0.0211
Epoch 4/10 | Loss: 0.0133
Epoch 5/10 | Loss: 0.0076
Epoch 6/10 | Loss: 0.0080
Epoch 7/10 | Loss: 0.0070
Epoch 8/10 | Loss: 0.0059
Epoch 9/10 | Loss: 0.0062
Epoch 10/10 | Loss: 0.0050
Training Time (sec): 175.7743604183197


In [ ]:
model.eval()
correct = 0
total = 0
total_spikes = 0

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        spk1, spk2 = model(images)
        output_sum = spk2.sum(dim=0)


        _, predicted = torch.max(output_sum, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        total_spikes += (
            spk1.sum() +
            spk2.sum()
        ).item()

test_acc = 100 * correct / total
print("Test Accuracy:", test_acc)
print("Total Spikes (All Layers):", total_spikes)

Test Accuracy: 98.71
Total Spikes (All Layers): 3271166.0


In [ ]:
metrics = {
    "model": "Hybrid",
    "accuracy": test_acc,
    "train_time_sec": train_time,
    "params": sum(p.numel() for p in model.parameters()),
    "flops": None,
    "spike_count": total_spikes
}

df = pd.DataFrame([metrics])
df.to_csv("hybrid_mnist_results.csv", index=False)

print("Saved hybrid_mnist_results.csv")

Saved hybrid_mnist_results.csv
